In [1]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# ── Configuration ──────────────────────────────────────────────────────────────
# Shared across all (domain, group) combinations — the JSON "methods" key is now
# uniformly "hscot" for both grid and mini domains (previously "two_stage" /
# "two_stage_cached"), so a single config + loop covers all 4 combinations below.
METHODS       = ["hscot", "random"]
COLORS        = {"hscot": "#F79646", "random": "#5a5a5a"}
HATCHES       = {"hscot": "",        "random": "///"}
DISPLAY_NAMES = {"hscot": "HSCOT",   "random": "Uniform Teaching"}

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

COMBINATIONS = [("grid", "single"), ("grid", "mixed"), ("mini", "single"), ("mini", "mixed")]

# ── Helpers ────────────────────────────────────────────────────────────────────
def safe_get_mean_heldout_regret(method_name, method_dict, domain):
    """Extract mean held-out regret, handling the key-path difference between domains."""
    if domain == "grid":
        # gridworld: methods.hscot.heldout.mean_regret
        #            methods.random.heldout.mean_regret
        heldout = method_dict.get("heldout", {})
        if heldout and heldout.get("mean_regret") is not None:
            return float(heldout["mean_regret"])
    else:
        # minigrid: methods.hscot.mean_heldout_regret
        #           methods.random.mean_heldout_regret
        val = method_dict.get("mean_heldout_regret")
        if val is not None:
            return float(val)
    return np.nan

def safe_get_coverage(method_name, method_dict):
    cs = method_dict.get("constraint_stats", {})
    if method_name == "hscot":
        return float(cs.get("coverage", cs.get("mean_coverage", np.nan)))
    if "mean_coverage" in cs:
        return float(cs["mean_coverage"])
    if "coverages" in cs and cs["coverages"]:
        return float(np.nanmean(cs["coverages"]))
    return np.nan

def safe_get_n_envs(method_name, method_dict):
    ss = method_dict.get("selection_stats", {})
    if method_name == "hscot":
        return float(ss.get("num_envs_used", np.nan))
    return float(ss.get("mean_mdp_count", np.nan))

# ── Load new-format results ─────────────────────────────────────────────────────
# New layout:  BASE_DIR/{modality}/seed_{N}/*.json
# One JSON per seed; feedback type is the parent directory name.

def load_new_format(base_dir, domain):
    """
    Returns a list of per-seed dicts:
      {"feedback": str, "seed": int, "{method}_regret": float, "{method}_cov": float}
    """
    records = []

    if not os.path.isdir(base_dir):
        print(f"[warn] Not found: {base_dir}")
        return records

    for modality in sorted(os.listdir(base_dir)):
        mod_dir = os.path.join(base_dir, modality)
        if not os.path.isdir(mod_dir):
            continue

        for seed_dir in sorted(os.listdir(mod_dir)):
            if not seed_dir.startswith("seed_"):
                continue
            seed_path = os.path.join(mod_dir, seed_dir)
            jsons = [f for f in os.listdir(seed_path) if f.endswith(".json")]
            if not jsons:
                # empty seed folder -> ignored, as intended
                continue

            # take the most recent file if multiple exist
            jsons.sort()
            json_path = os.path.join(seed_path, jsons[-1])

            try:
                with open(json_path) as f:
                    data = json.load(f)
            except Exception as e:
                print(f"[warn] Could not read {json_path}: {e}")
                continue

            methods_data = data.get("methods", {})
            rec = {"feedback": modality, "seed": seed_dir}

            for m in METHODS:
                md = methods_data.get(m, {})
                rec[f"{m}_regret"] = safe_get_mean_heldout_regret(m, md, domain)
                rec[f"{m}_cov"]    = safe_get_coverage(m, md)
                rec[f"{m}_n_envs"] = safe_get_n_envs(m, md)

            records.append(rec)

    return records

# ────────────────────────────────────────────────
# Plotting
# ────────────────────────────────────────────────

def clean_feedback_label(fb):
    fb = str(fb).lower()
    if fb == "correction":
        return "Correction"
    if fb == "pairwise":
        return "Comparison"
    if fb == "demo_estop":
        return "Demo+\nEstop"
    if fb == "demo_correction":
        return "Demo+\nCorrection"
    if fb == "demo_pairwise":
        return "Demo+\nComparison"
    return fb.capitalize()

def plot_grouped(feedbacks, stats, fig_prefix, metric_suffix, ylabel):
    x = np.arange(len(feedbacks))
    width = 0.38

    display_feedbacks = [clean_feedback_label(fb) for fb in feedbacks]

    plt.figure(figsize=(10.8, 7.2))

    for i, m in enumerate(METHODS):
        means = [stats[fb][f"{m}_{metric_suffix}_mean"] for fb in feedbacks]
        stds = [stats[fb][f"{m}_{metric_suffix}_std"] for fb in feedbacks]

        plt.bar(
            x + (i - 0.5) * width,
            means,
            width,
            yerr=stds,
            capsize=6,
            label=DISPLAY_NAMES[m],
            color=COLORS[m],
            edgecolor="black",
            linewidth=0.9,
            hatch=HATCHES[m],
            error_kw={"elinewidth": 1.4, "capthick": 1.4}
        )

    plt.xticks(x, display_feedbacks, fontsize=33, rotation=00)
    plt.yticks(fontsize=28)
    plt.ylabel(ylabel, fontsize=33)
    plt.grid(axis="y", linestyle="--", alpha=0.35)
    plt.legend(
        frameon=True,
        fontsize=33,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.2),
        ncol=2
    )

    plt.tight_layout(pad=0.3)
    base_name = f"{fig_prefix}_{metric_suffix}"

    pdf_path = os.path.join(FIG_DIR, base_name + ".pdf")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.close()

    print(f"Saved: {pdf_path}")

# ────────────────────────────────────────────────
# Run for all 4 (domain, group) combinations
# ────────────────────────────────────────────────

for DOMAIN, GROUP in COMBINATIONS:
    BASE_DIR   = f"./{DOMAIN}_{GROUP}"
    FIG_PREFIX = f"{DOMAIN}_50envs" if GROUP == "single" else f"{DOMAIN}_tie_50envs"

    print(f"\n=== {DOMAIN}_{GROUP} ({BASE_DIR}) ===")

    all_records = load_new_format(BASE_DIR, DOMAIN)

    if not all_records:
        print("No valid results found.")
        continue

    print(f"Loaded {len(all_records)} seed records across "
          f"{len(set(r['feedback'] for r in all_records))} modalities")

    # ── Aggregate across seeds ──────────────────────────────────────────────────
    grouped = defaultdict(lambda: defaultdict(list))
    for r in all_records:
        fb = r["feedback"]
        for m in METHODS:
            grouped[fb][f"{m}_regret"].append(r[f"{m}_regret"])
            grouped[fb][f"{m}_cov"].append(r[f"{m}_cov"])

    feedbacks = sorted(grouped.keys())
    stats = defaultdict(dict)
    for fb in feedbacks:
        for m in METHODS:
            r_vals = np.array(grouped[fb][f"{m}_regret"], dtype=float)
            c_vals = np.array(grouped[fb][f"{m}_cov"], dtype=float)

            # SEM denominator counts only non-NaN seeds, not len(), so a missing
            # value for one seed doesn't inflate N and shrink the SEM.
            n_r = max(np.sum(~np.isnan(r_vals)), 1)
            n_c = max(np.sum(~np.isnan(c_vals)), 1)

            stats[fb][f"{m}_regret_mean"] = np.nanmean(r_vals)
            stats[fb][f"{m}_regret_std"]  = np.nanstd(r_vals) / np.sqrt(n_r)  # SEM
            stats[fb][f"{m}_cov_mean"]    = np.nanmean(c_vals)
            stats[fb][f"{m}_cov_std"]     = np.nanstd(c_vals) / np.sqrt(n_c)

    plot_grouped(feedbacks, stats, FIG_PREFIX, "regret", "Held-Out Regret")
    plot_grouped(feedbacks, stats, FIG_PREFIX, "cov", "Mean Coverage")



=== grid_single (./grid_single) ===
Loaded 39 seed records across 4 modalities
Saved: figures/grid_50envs_regret.pdf


Saved: figures/grid_50envs_cov.pdf

=== grid_mixed (./grid_mixed) ===
Loaded 29 seed records across 3 modalities
Saved: figures/grid_tie_50envs_regret.pdf


Saved: figures/grid_tie_50envs_cov.pdf

=== mini_single (./mini_single) ===
Loaded 40 seed records across 4 modalities
Saved: figures/mini_50envs_regret.pdf


Saved: figures/mini_50envs_cov.pdf

=== mini_mixed (./mini_mixed) ===
Loaded 30 seed records across 3 modalities
Saved: figures/mini_tie_50envs_regret.pdf
Saved: figures/mini_tie_50envs_cov.pdf


In [2]:
import os, json, glob
import numpy as np
import pandas as pd
from collections import defaultdict

# ── Config ─────────────────────────────────────────────────────────────────────
BASE_DIR = "./mini_single"          # relative to notebook location (results_chpc/)

# JSON method keys in the mini domain
METHOD_KEY_SCOT   = "hscot"
METHOD_KEY_RANDOM = "random"

FEEDBACK_ORDER  = ["demo", "pairwise", "estop", "correction"]
FEEDBACK_LABELS = {"demo": "Demo", "pairwise": "Comparison",
                   "estop": "E-stop", "correction": "Correction"}

# ── Extractors ─────────────────────────────────────────────────────────────────
def get_heldout_regret(md):
    v = md.get("mean_heldout_regret")
    return float(v) if v is not None else np.nan

def get_coverage(method_key, md):
    cs = md.get("constraint_stats", {})
    if method_key == METHOD_KEY_SCOT:
        return float(cs["coverage"]) if "coverage" in cs else np.nan
    # random: averaged over trials
    if "mean_coverage" in cs:
        return float(cs["mean_coverage"])
    if "coverages" in cs and cs["coverages"]:
        return float(np.mean(cs["coverages"]))
    return np.nan

def get_n_envs(method_key, md):
    ss = md.get("selection_stats", {})
    if method_key == METHOD_KEY_SCOT:
        return float(ss["num_envs_used"]) if "num_envs_used" in ss else np.nan
    return float(ss["mean_mdp_count"]) if "mean_mdp_count" in ss else np.nan

# ── Load ───────────────────────────────────────────────────────────────────────
per_seed = defaultdict(lambda: defaultdict(list))   # [feedback][metric] -> [values]

for feedback in FEEDBACK_ORDER:
    mod_dir = os.path.join(BASE_DIR, feedback)
    if not os.path.isdir(mod_dir):
        continue
    for seed_dir in sorted(os.listdir(mod_dir)):
        if not seed_dir.startswith("seed_"):
            continue
        jsons = sorted(glob.glob(os.path.join(mod_dir, seed_dir, "*.json")))
        if not jsons:
            continue
        with open(jsons[-1]) as f:
            data = json.load(f)
        methods = data.get("methods", {})

        for mkey in [METHOD_KEY_SCOT, METHOD_KEY_RANDOM]:
            if mkey not in methods:
                continue
            md = methods[mkey]
            per_seed[feedback][f"{mkey}_regret"].append(get_heldout_regret(md))
            per_seed[feedback][f"{mkey}_coverage"].append(get_coverage(mkey, md))
            per_seed[feedback][f"{mkey}_n_envs"].append(get_n_envs(mkey, md))

# ── Aggregate ──────────────────────────────────────────────────────────────────
def fmt(vals, decimals=4):
    arr = np.array(vals, dtype=float)
    n   = np.sum(~np.isnan(arr))
    if n == 0:
        return "—"
    mean = np.nanmean(arr)
    sem  = np.nanstd(arr) / np.sqrt(n) if n > 1 else 0.0
    return f"{mean:.{decimals}f} ± {sem:.{decimals}f}"

rows = []
for fb in FEEDBACK_ORDER:
    if fb not in per_seed:
        continue
    d = per_seed[fb]
    n_seeds = len(d.get(f"{METHOD_KEY_SCOT}_regret", []))
    rows.append({
        "Feedback"          : FEEDBACK_LABELS[fb],
        #"Seeds"             : n_seeds,
        #"HSCOT Regret"      : fmt(d[f"{METHOD_KEY_SCOT}_regret"]),
        #"HSCOT Coverage"    : fmt(d[f"{METHOD_KEY_SCOT}_coverage"], decimals=3),
        "HSCOT #Envs"       : fmt(d[f"{METHOD_KEY_SCOT}_n_envs"],   decimals=1),
        #"Random Regret"     : fmt(d[f"{METHOD_KEY_RANDOM}_regret"]),
        #"Random Coverage"   : fmt(d[f"{METHOD_KEY_RANDOM}_coverage"], decimals=3),
        "Random #Envs"      : fmt(d[f"{METHOD_KEY_RANDOM}_n_envs"],   decimals=1),
    })

df = pd.DataFrame(rows)
#print("MiniGrid — held-out regret, coverage, #envs   (mean ± SEM across seeds)\n")
print(df.to_string(index=False))



  Feedback HSCOT #Envs Random #Envs
      Demo   5.5 ± 0.4    7.1 ± 0.4
Comparison   7.5 ± 0.3    7.9 ± 0.4
    E-stop  12.0 ± 0.4   15.0 ± 0.5
Correction   6.9 ± 0.6    7.0 ± 0.5
